In [0]:
df_slv_orders = spark.read.format("delta").load("s3://sj-dbr-demo-proj/silver_data/slv_orders/")
df_slv_orders.show(3)

##### Lets ad new columns to SIlver DF before writing to gold

In [0]:
from pyspark.sql.functions import col, lit, when
from pyspark.sql.types import DoubleType

df_slv_orders = df_slv_orders.withColumns({
    "gross_amt": col("quantity") * col("unit_price").cast(DoubleType()),
    "discount_amt": (col("quantity") * col("unit_price").cast(DoubleType())) * (col("discount_pct") / 100),
    "sale_amt": (col("quantity") * col("unit_price").cast(DoubleType())) - 
                ((col("quantity") * col("unit_price").cast(DoubleType())) * (col("discount_pct") / 100)),
    "coupon_flag": when(col("coupon_code").isNotNull(), lit(1)).otherwise(lit(0))
})

df_slv_orders.limit(3).display()

Lets create all sales amount into INR format

In [0]:
df_slv_orders.select(col("unit_price_currency")).distinct().show()


In [0]:
from pyspark.sql import Row
rates = [] 
currency_rates = {
    "INR" : 1.00,
    "USD" : 88.29,
    "SGD" : 68.16,
    "GBP" : 117.90,
    "CAD" : 62.93,
    "AED" : 24.18,
    "AUD" : 57.55
}
for c,v in currency_rates.items():
    rates.append(Row(currency=c, value= v))

currency_df = spark.createDataFrame(rates)

In [0]:
df = df_slv_orders.join(
    currency_df,
    df_slv_orders.unit_price_currency == currency_df.currency,
    "left"
)
df = df.withColumn("price_in_inr", col("sale_amt") * col("value"))

col_select_list = ['dt',
 'order_ts',
 'customer_id',
 'order_id',
 'item_seq',
 'product_id',
 'quantity',
 'unit_price_currency',
 'unit_price',
 'discount_pct',
 'tax_amount',
 'channel',
 'coupon_code',
 'gross_amt',
 'discount_amt',
 'sale_amt',
 'coupon_flag',
 'price_in_inr']

df = df.select(col_select_list)
df.limit(3).display()

#### WRITE TO GOLD LAYER

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", True).save("s3://sj-dbr-demo-proj/gold_data/fact_orders")

spark.sql("""
         create table if not exists ecommerce.gold.fact_orders 
         using delta 
         location 's3://sj-dbr-demo-proj/gold_data/fact_orders'
         """)

In [0]:
dbutils.notebook.exit('SUCCESS')